# Level 13 — Covariance Estimation

**Audience:** analysts who already understand sample volatility and portfolio
variance.

**Prerequisites:** Levels 1–2, NumPy, pandas, and labelled return data.

**Learning goals**

1. estimate a labelled sample covariance matrix;
2. construct a constant-correlation target;
3. blend the two with an explicit shrinkage intensity;
4. show how covariance choice changes GMV weights.

This tutorial independently implements public covariance-estimation concepts
using synthetic monthly returns. It contains no course data or network access.

## 1. Setup and synthetic observations

All three assets use the same 72 monthly observations. The random seed makes
the example exactly reproducible.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.estimation import (
    constant_correlation_covariance,
    sample_covariance,
    shrink_covariance,
)
from asset_management_toolkit.portfolio import global_minimum_variance

In [ ]:
rng = np.random.default_rng(20260728)
dates = pd.date_range("2020-01-31", periods=72, freq="ME")
monthly_returns = pd.DataFrame(
    rng.multivariate_normal(
        mean=[0.006, 0.002, 0.004],
        cov=[
            [0.0025, 0.0002, 0.0005],
            [0.0002, 0.0004, 0.0001],
            [0.0005, 0.0001, 0.0012],
        ],
        size=len(dates),
    ),
    index=dates,
    columns=["Equity", "Bonds", "Real assets"],
)
monthly_returns.describe().loc[["mean", "std"]]

## 2. Sample covariance

`sample_covariance` uses one complete observation set for every pair. Its
default `ddof=1` matches the usual sample covariance convention.

In [ ]:
sample = sample_covariance(monthly_returns)
sample

## 3. Constant-correlation target

The target keeps each sample variance on the diagonal. It replaces all
off-diagonal correlations with their cross-sectional average.

In [ ]:
constant_correlation = constant_correlation_covariance(monthly_returns)

def covariance_to_correlation(covariance):
    volatility = np.sqrt(np.diag(covariance))
    return covariance / np.outer(volatility, volatility)

pd.DataFrame(
    covariance_to_correlation(constant_correlation),
    index=constant_correlation.index,
    columns=constant_correlation.columns,
)

## 4. Caller-controlled shrinkage

An intensity of 0.40 gives 40% weight to the structured target and 60% to the
sample estimate. It is a chosen scenario, not an estimated optimal
Ledoit–Wolf coefficient.

In [ ]:
shrunk = shrink_covariance(monthly_returns, intensity=0.40)
pd.DataFrame(
    {
        "sample": sample.stack(),
        "constant_correlation": constant_correlation.stack(),
        "shrunk_40_percent": shrunk.stack(),
    }
).head(9)

## 5. Portfolio sensitivity

GMV construction can amplify covariance-estimation differences. Comparing
weights makes model risk visible rather than hiding it inside one matrix.

In [ ]:
gmv_comparison = pd.concat(
    {
        "Sample": global_minimum_variance(sample),
        "Constant correlation": global_minimum_variance(constant_correlation),
        "40% shrinkage": global_minimum_variance(shrunk),
    },
    axis=1,
)
gmv_comparison

## Exercise — inspect the shrinkage path

Calculate GMV weights at intensities 0, 0.25, 0.50, 0.75, and 1. Which asset's
weight is most sensitive to the covariance assumption?

In [ ]:
# Try it here.
intensities = [0.0, 0.25, 0.50, 0.75, 1.0]

### Answer scaffold

In [ ]:
pd.DataFrame(
    {
        intensity: global_minimum_variance(
            shrink_covariance(monthly_returns, intensity=intensity)
        )
        for intensity in intensities
    }
).T.rename_axis("shrinkage_intensity")

## Interpretation and pitfalls

- Covariance estimates depend on the sampling window and return frequency.
- Missing values are rejected to avoid inconsistent pairwise sample sizes.
- A constant-correlation target is structured, not automatically correct.
- Fixed intensity is a sensitivity parameter, not an optimal estimate.
- Portfolio-weight stability and chronological holdout risk matter more than
  in-sample matrix fit.

Next: use an estimated covariance matrix to construct explicit risk budgets.